# Notebook 4 — LLM-as-a-Judge con mitigación de sesgos
**Módulo M2 — Evaluación de NER clínico**

Este notebook implementa la dimensión de evaluación cualitativa con un **LLM como juez clínico**, complementando las métricas de exact-match (Dimensión 1) y similitud semántica (Dimensión 2):
1. **Carga y evaluación del Gold Set** (`gold_examples.jsonl`) con el modelo fine-tuneado (`clinical_BERT + LoRA`).
2. **Construcción de Rich Examples** estandarizada y compatible con el harness unificado del equipo.
3. **Juez LLM (`openai/gpt-oss-120b`)** con rúbrica clínica de 4 dimensiones (completitud, exactitud de boundary, relevancia clínica, ausencia de ruido).
4. **Mitigación rigurosa de sesgos:**
   - **Sesgo de posición:** Doble evaluación (normal + invertido) y mitigación por promedio.
   - **Sesgo de longitud:** Pares de validación controlados (Tipo A y Tipo B).
   - **Sesgo de auto-preferencia:** Anonimización total del output y uso de familia externa.
5. **Scorecard y triangulación con F1 exacto:** Detección de errores de span vs errores de comprensión clínica.

## 0. Setup e imports

In [3]:
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
    print("Ejecutando en Google Colab — Drive montado.")
except ImportError:
    IN_COLAB = False
    print("Ejecutando en entorno local.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ejecutando en Google Colab — Drive montado.


In [4]:
import subprocess, sys

required = ["groq", "datasets", "peft", "transformers", "accelerate"]
installed = []

for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        installed.append(pkg)

if installed:
    print(f"Instalando: {installed}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + installed)
    print("Instalación completada.")
else:
    print("Todas las dependencias ya están disponibles.")
# Desinstalar torchao si existe para evitar el bug de versión con peft en Colab
try:
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
except Exception:
    pass


Todas las dependencias ya están disponibles.


In [5]:
import re, json, random, time
from pathlib import Path
import itertools
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from peft import PeftModel

from groq import Groq

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Seeds fijadas (SEED={SEED}). Device: {device}")

import transformers as _tr, peft as _peft
print(f"  transformers: {_tr.__version__}")
print(f"  peft:         {_peft.__version__}")
print(f"  torch:        {torch.__version__}")
print(f"  numpy:        {np.__version__}")
# Evitar incompatibilidad de versión de torchao en Colab
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass


Seeds fijadas (SEED=42). Device: cuda
  transformers: 5.16.1
  peft:         0.20.0
  torch:        2.11.0+cu128
  numpy:        2.1.3


In [6]:
# ============================================================
#  CONFIGURACIÓN DEL PROYECTO (Rutas sincronizadas con el harness)
# ============================================================

if IN_COLAB:
    _drive_root = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
    if not _drive_root.exists():
        _drive_root = Path("/content/drive/MyDrive/TopicosIA")
    PROJECT_ROOT = _drive_root
else:
    PROJECT_ROOT = Path("..") if Path("../M1").exists() else Path(".")

# Adaptador LoRA de Clinical BERT
MODEL_DIR = PROJECT_ROOT / "M1" / "saved_models" / "clinical_bert-distemist-lora"

# Gold set generado para evaluación de harness (formato JSONL)
# Búsqueda flexible de gold_examples.jsonl (en M2/eval_harness o directamente en M2)
_gold_candidates = [
    PROJECT_ROOT / "M2" / "eval_harness" / "gold_examples.jsonl",
    PROJECT_ROOT / "M2" / "gold_examples.jsonl",
    PROJECT_ROOT / "eval_harness" / "gold_examples.jsonl",
    PROJECT_ROOT / "gold_examples.jsonl",
    Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/M2/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/Topicos-IA/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/Topicos-IA/M2/gold_examples.jsonl"),
    Path("M2/eval_harness/gold_examples.jsonl"),
    Path("M2/gold_examples.jsonl"),
    Path("gold_examples.jsonl"),
]
GOLD_SET_PATH = next((p for p in _gold_candidates if p.exists()), PROJECT_ROOT / "M2" / "gold_examples.jsonl")

# Directorio de salidas de M2
OUTPUT_DIR = PROJECT_ROOT / "M2" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ruta al resultado de dimensión 1 (exact match)
DIM1_PATH = OUTPUT_DIR / "resultado_dimension1.json"

# Checkpoint base oficial del proyecto
BASE_CHECKPOINT = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"

# ============================================================
#  CONFIGURACIÓN DEL JUEZ LLM (Groq)
# ============================================================
GROQ_API_KEY = ""   # pegar la API key aquí (gratis en https://console.groq.com/keys) o configurar en Colab Secrets (GROQ_API_KEY)

# Modelo fijado para el juez (reproducibilidad unificada con el harness)
JUDGE_MODEL = "openai/gpt-oss-120b"

JUDGE_TEMPERATURE = 0.0   # determinístico
JUDGE_MAX_TOKENS  = 1500  # suficiente para respuestas con razonamiento

# Parámetros de chunking (M1)
WINDOW_WORDS  = 277
OVERLAP_WORDS = 50

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"MODEL_DIR:     {MODEL_DIR}")
print(f"GOLD_SET_PATH: {GOLD_SET_PATH} (existe: {GOLD_SET_PATH.exists()})")
print(f"OUTPUT_DIR:    {OUTPUT_DIR}")
print(f"JUDGE_MODEL:   {JUDGE_MODEL}")

# Si el adaptador no existe aún en local/drive, soporte de descarga directa desde la rama
if not (MODEL_DIR / "adapter_model.safetensors").exists():
    print(f"[INFO] Modelo LoRA no encontrado en {MODEL_DIR}. Intentando clonar desde rama clinical-bert...")
    try:
        import subprocess, shutil
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            "git", "clone", "--depth", "1", "-b", "clinical-bert",
            "https://github.com/luisNP21/Topicos-IA.git", "/tmp/repo_clinical"
        ], check=True)
        for item in Path("/tmp/repo_clinical/M1/clinical_BERT/clinical_bert-distemist-lora").glob("*"):
            shutil.copy2(item, MODEL_DIR)
        print(f"[OK] Modelo LoRA descargado y listo en {MODEL_DIR}")
    except Exception as e:
        print(f"[AVISO] No se pudo descargar automáticamente: {e}. Verificar ruta en Drive.")
else:
    print(f"[OK] Modelo LoRA verificado en {MODEL_DIR}")

PROJECT_ROOT:  /content/drive/MyDrive/TopicosIA
MODEL_DIR:     /content/drive/MyDrive/TopicosIA/M1/saved_models/clinical_bert-distemist-lora
GOLD_SET_PATH: /content/drive/MyDrive/TopicosIA/M2/gold_examples.jsonl (existe: True)
OUTPUT_DIR:    /content/drive/MyDrive/TopicosIA/M2/outputs
JUDGE_MODEL:   openai/gpt-oss-120b
[OK] Modelo LoRA verificado en /content/drive/MyDrive/TopicosIA/M1/saved_models/clinical_bert-distemist-lora


In [7]:
import os
from groq import Groq

# Recuperar API Key desde variable de celda, Colab Secrets o entorno
api_key = GROQ_API_KEY
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        pass
if not api_key:
    api_key = os.environ.get("GROQ_API_KEY", "")

if not api_key:
    raise ValueError(
        "No se encontró la API key de Groq. "
        "Obtén una gratis en https://console.groq.com/keys y pégala en GROQ_API_KEY "
        "o agrégala a Colab Secrets como GROQ_API_KEY."
    )

groq_client = Groq(api_key=api_key)

def test_model_ping(model_name: str) -> bool:
    try:
        groq_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "hola"}],
            max_tokens=10,
        )
        return True
    except Exception:
        return False

# Verificación directa sin fallback automático para garantizar reproducibilidad exacta
print(f"Verificando disponibilidad de JUDGE_MODEL = '{JUDGE_MODEL}' en Groq...")
if not test_model_ping(JUDGE_MODEL):
    err_msg = (
        f"El modelo juez '{JUDGE_MODEL}' no está disponible en Groq (probablemente deprecado o sin acceso).\n"
        f"Revisar https://console.groq.com/docs/deprecations y actualizar JUDGE_MODEL manualmente.\n"
        f"NO se aplica fallback automático para preservar reproducibilidad entre ejecuciones."
    )
    raise RuntimeError(err_msg)
else:
    print(f"[OK] Modelo juez configurado '{JUDGE_MODEL}' verificado exitosamente.")

# Comprobar si el modelo soporta response_format={'type': 'json_object'}
SUPPORTS_JSON_MODE = True
try:
    groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": "Responde solo JSON."},
            {"role": "user", "content": 'Genera {"ping": "pong"}'},
        ],
        max_tokens=25,
        response_format={"type": "json_object"},
    )
    print("  Soporte de JSON mode nativo (response_format): SI")
except Exception:
    SUPPORTS_JSON_MODE = False
    print("  Soporte de JSON mode nativo (response_format): NO (se usará parseo robusto por regex)")

print(f"\nJuez LLM listo para evaluación: {JUDGE_MODEL} (Groq API)")

Verificando disponibilidad de JUDGE_MODEL = 'openai/gpt-oss-120b' en Groq...
[OK] Modelo juez configurado 'openai/gpt-oss-120b' verificado exitosamente.
  Soporte de JSON mode nativo (response_format): NO (se usará parseo robusto por regex)

Juez LLM listo para evaluación: openai/gpt-oss-120b (Groq API)


---
## 1. Cargar el Gold Set del equipo

El gold set es un archivo JSONL (`gold_examples.jsonl`) generado de manera centralizada en M1 a partir del split `test`.
Cada ejemplo contiene:
- `input`: texto clínico del documento.
- `esperado`: lista de entidades patológicas anotadas (`list[str]`).
- `criterio`: justificación de selección.

In [8]:
import zipfile
import xml.etree.ElementTree as ET
import ast

def load_gold_from_docx(docx_path: Path) -> list:
    """Extrae los ejemplos gold estructurados desde el archivo Word Ejemplos_Gold.docx."""
    with zipfile.ZipFile(docx_path) as z:
        xml_content = z.read("word/document.xml")
        tree = ET.fromstring(xml_content)
        text = [elem.text for elem in tree.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t") if elem.text]
        full = "".join(text)
    matches = re.findall(r"(\{[^{}]*?'input'[\s\S]*?\})", full)
    examples = []
    for m in matches:
        s = m.strip()
        if s.endswith(","):
            s = s[:-1]
        s = re.sub(r"(['\"])\s*('esperado')", r"\1, \2", s)
        s = re.sub(r"(['\"])\s*('criterio')", r"\1, \2", s)
        s = re.sub(r"(\])\s*('criterio')", r"\1, \2", s)
        try:
            d = ast.literal_eval(s)
            examples.append(d)
        except Exception:
            pass
    return examples

# Búsqueda inteligente del gold set (JSONL o DOCX en Drive y local)
candidate_paths = [
    GOLD_SET_PATH,
    PROJECT_ROOT / "M2" / "eval_harness" / "gold_examples.jsonl",
    PROJECT_ROOT / "M2" / "gold_examples.jsonl",
    Path("M2/eval_harness/gold_examples.jsonl"),
    Path("M2/gold_examples.jsonl"),
    Path("eval_harness/gold_examples.jsonl"),
    Path("gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/Topicos-IA/M2/eval_harness/gold_examples.jsonl"),
]

resolved_gold_path = next((p for p in candidate_paths if p.exists()), None)
gold_examples = []

if resolved_gold_path:
    print(f"Cargando gold set desde JSONL: {resolved_gold_path}")
    with open(resolved_gold_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                gold_examples.append(json.loads(line))
else:
    # Buscar Ejemplos_Gold.docx
    candidate_docx = [
        PROJECT_ROOT / "M2" / "Ejemplos_Gold.docx",
        PROJECT_ROOT / "Ejemplos_Gold.docx",
        Path("M2/Ejemplos_Gold.docx"),
        Path("Ejemplos_Gold.docx"),
        Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/Ejemplos_Gold.docx"),
        Path("/content/drive/MyDrive/TopicosIA/M2/Ejemplos_Gold.docx"),
        Path("/content/drive/MyDrive/Topicos-IA/M2/Ejemplos_Gold.docx"),
    ]
    docx_path = next((p for p in candidate_docx if p.exists()), None)
    if docx_path:
        print(f"Extrayendo gold set desde archivo Word de ejemplos: {docx_path}")
        gold_examples = load_gold_from_docx(docx_path)
        # Sincronizar y persistir automáticamente en formato JSONL para el harness
        GOLD_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(GOLD_SET_PATH, "w", encoding="utf-8") as f:
            for ex in gold_examples:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")
        print(f"Guardado como JSONL en: {GOLD_SET_PATH}")
    else:
        # Descargar directamente del repositorio GitHub si no está en Drive
        print("[INFO] Buscando gold_examples.jsonl en repositorio GitHub...")
        try:
            import urllib.request
            url = "https://raw.githubusercontent.com/luisNP21/Topicos-IA/Agustin/M2/eval_harness/gold_examples.jsonl"
            GOLD_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, str(GOLD_SET_PATH))
            with open(GOLD_SET_PATH, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        gold_examples.append(json.loads(line))
            print(f"[OK] Descargado exitosamente desde GitHub: {len(gold_examples)} ejemplos.")
        except Exception as e:
            raise FileNotFoundError(
                f"No se encontró gold_examples.jsonl ni Ejemplos_Gold.docx en Drive ni local.\n"
                f"Ubicaciones verificadas: {[str(p) for p in candidate_paths]}\n"
                f"Error al descargar: {e}"
            )

print(f"\nGold set cargado: {len(gold_examples)} ejemplos")
assert len(gold_examples) > 0, "El gold set está vacío"
assert "input" in gold_examples[0], "Falta campo 'input' en el gold set"
assert "esperado" in gold_examples[0], "Falta campo 'esperado' en el gold set"
assert isinstance(gold_examples[0]["esperado"], list), (
    f"'esperado' debe ser list[str], vino como {type(gold_examples[0]['esperado'])}"
)

print("Contrato verificado exitosamente.")
print("Primer ejemplo:")
print("  input[:120]:", gold_examples[0]["input"][:120], "...")
print("  esperado:", gold_examples[0]["esperado"][:5])
print("  n_entidades:", len(gold_examples[0]["esperado"]))

Cargando gold set desde JSONL: /content/drive/MyDrive/TopicosIA/M2/gold_examples.jsonl

Gold set cargado: 59 ejemplos
Contrato verificado exitosamente.
Primer ejemplo:
  input[:120]: Varón soltero de 37 años de edad, trabajador del campo, que es enviado a nuestro Servicio desde otro hospital por no ten ...
  esperado: ['adenopatías', 'adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales', 'adenopatías bilaterales ilíacas e inguinales', 'afectación de las cadenas ganglionares', 'brucelosis']
  n_entidades: 28


---
## 2. Cargar el modelo fine-tuneado (Clinical BERT + LoRA)

Carga del checkpoint base `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es` e inyección de los adaptadores LoRA entrenados en M1.

In [9]:
label_list = ["O", "B-ENFERMEDAD", "I-ENFERMEDAD"]
id2label   = {i: l for i, l in enumerate(label_list)}
label2id   = {l: i for i, l in enumerate(label_list)}

base_model = AutoModelForTokenClassification.from_pretrained(
    BASE_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)


# Parche definitivo contra el bug de versión de torchao en Google Colab
import sys
import peft.import_utils
from packaging import version
peft.import_utils.TORCHAO_MINIMUM_VERSION = version.parse("0.0.1")
peft.import_utils.is_torchao_available = lambda: False
for mod in list(sys.modules.values()):
    if hasattr(mod, "TORCHAO_MINIMUM_VERSION"):
        try:
            mod.TORCHAO_MINIMUM_VERSION = version.parse("0.0.1")
        except Exception:
            pass
    if hasattr(mod, "is_torchao_available"):
        try:
            mod.is_torchao_available = lambda: False
        except Exception:
            pass

model = PeftModel.from_pretrained(base_model, str(MODEL_DIR))
model.eval()   # Desactiva dropout para inferencia determinista

if torch.cuda.is_available():
    model = model.to("cuda")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))

print("Modelo base y adaptador LoRA cargados exitosamente.")
print("Dispositivo:", next(model.parameters()).device)
print("Vocab size:", tokenizer.vocab_size)

# Sanity check con frase médica
frase_prueba = "El paciente presenta diabetes mellitus tipo 2 y antecedentes de hipertension arterial."
tokens_prueba = frase_prueba.split()

inputs_p = tokenizer(
    tokens_prueba, is_split_into_words=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    logits_p = model(**inputs_p).logits

pred_ids_p = logits_p.argmax(dim=-1)[0].tolist()
word_ids_p = inputs_p.word_ids(batch_index=0)

print("\nSanity check -- predicciones por palabra:")
seen = set()
for idx, wid in zip(pred_ids_p, word_ids_p):
    if wid is None or wid in seen:
        continue
    print(f"  {tokens_prueba[wid]:<25} -> {id2label[idx]}")
    seen.add(wid)

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  504MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  504MB            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Modelo base y adaptador LoRA cargados exitosamente.
Dispositivo: cuda:0
Vocab size: 52000

Sanity check -- predicciones por palabra:
  El                        -> O
  paciente                  -> O
  presenta                  -> O
  diabetes                  -> B-ENFERMEDAD
  mellitus                  -> I-ENFERMEDAD
  tipo                      -> I-ENFERMEDAD
  2                         -> I-ENFERMEDAD
  y                         -> O
  antecedentes              -> O
  de                        -> O
  hipertension              -> B-ENFERMEDAD
  arterial.                 -> I-ENFERMEDAD


---
## 3. Funciones de predicción y chunking de M1

Mismas funciones estándar de `harness.ipynb` para segmentación en ventanas con overlap y agregación determinista por documento.

In [10]:
import string

def normalizar_entidad(e):
    """Quita puntuación de borde que es artefacto de tokenizar con .split()."""
    return e.lower().strip().strip(string.punctuation + " ")


def chunk_words(words, window_words, overlap_words):
    """Parte una lista de palabras en ventanas solapadas."""
    if len(words) <= window_words:
        return [(0, words)]

    chunks = []
    step = window_words - overlap_words
    start = 0
    idx = 0
    while start < len(words):
        chunk = words[start : start + window_words]
        chunks.append((idx, chunk))
        if start + window_words >= len(words):
            break
        start += step
        idx += 1
    return chunks


def strip_chunk_suffix(doc_id):
    return re.sub(r"_chunk\d+$", "", doc_id)


def bio_to_entity_set(tokens, tags):
    entities, current = [], []
    for tok, tag in zip(tokens, tags):
        if tag == "B-ENFERMEDAD":
            if current:
                entities.append(" ".join(current))
            current = [tok]
        elif tag == "I-ENFERMEDAD" and current:
            current.append(tok)
        else:
            if current:
                entities.append(" ".join(current))
            current = []
    if current:
        entities.append(" ".join(current))
    return set(normalizar_entidad(e) for e in entities)


def aggregate_entities_by_original_doc(doc_ids, entity_sets):
    grouped = {}
    for doc_id, ents in zip(doc_ids, entity_sets):
        orig_id = strip_chunk_suffix(doc_id)
        grouped.setdefault(orig_id, set()).update(ents)
    return grouped


def micro_prf1_by_doc(true_by_doc, pred_by_doc):
    tp = fp = fn = 0
    all_doc_ids = set(true_by_doc) | set(pred_by_doc)
    for doc_id in all_doc_ids:
        true_ents = true_by_doc.get(doc_id, set())
        pred_ents = pred_by_doc.get(doc_id, set())
        tp += len(true_ents & pred_ents)
        fp += len(pred_ents - true_ents)
        fn += len(true_ents - pred_ents)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn}


def predict_word_level(tokens):
    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
    ).to(model.device)
    word_ids = inputs.word_ids(batch_index=0)

    with torch.no_grad():
        logits = model(**inputs).logits
    pred_ids = logits.argmax(dim=-1)[0].tolist()

    word_preds, seen = [], set()
    for idx, wid in zip(pred_ids, word_ids):
        if wid is None or wid in seen:
            continue
        word_preds.append(id2label[idx])
        seen.add(wid)
    return word_preds


def predict_entities_chunked(text, window_words, overlap_words):
    words = text.split()
    chunks = chunk_words(words, window_words, overlap_words)

    resultados = []
    for chunk_idx, chunk_words_list in chunks:
        pred_tags = predict_word_level(chunk_words_list)
        entity_set = bio_to_entity_set(chunk_words_list, pred_tags)
        suffix = f"_chunk{chunk_idx}" if len(chunks) > 1 else ""
        resultados.append((suffix, entity_set))
    return resultados


def dimension_metrica_clasica(gold_examples, window_words, overlap_words):
    true_by_doc = {}
    pred_doc_ids, pred_entity_sets = [], []

    for i, ex in enumerate(gold_examples):
        doc_id = f"ex_{i}"
        true_by_doc[doc_id] = {normalizar_entidad(e) for e in ex["esperado"]}

        resultados_chunks = predict_entities_chunked(ex["input"], window_words, overlap_words)
        for suffix, entity_set in resultados_chunks:
            pred_doc_ids.append(doc_id + suffix)
            pred_entity_sets.append(entity_set)

    pred_by_doc = aggregate_entities_by_original_doc(pred_doc_ids, pred_entity_sets)
    metrics = micro_prf1_by_doc(true_by_doc, pred_by_doc)
    return metrics, true_by_doc, pred_by_doc

print("Funciones de inferencia y evaluación cargadas.")

Funciones de inferencia y evaluación cargadas.


---
## 4. Formato y preparación de Rich Examples para el Juez

Cargamos los ejemplos directamente desde `resultado_dimension1.json` (`DIM1_PATH`), garantizando que:
1. Usamos exactamente los mismos ejemplos del gold set que la Dimensión 1 y la Dimensión 2 (sin discrepancias de subset).
2. Si `DIM1_PATH` ya existe, se lee instantáneamente; si no existe, se calcula ejecutando `dimension_metrica_clasica` sobre `gold_examples`.

In [11]:
if DIM1_PATH.exists():
    print(f"Cargando predicciones existentes desde: {DIM1_PATH}")
    with open(DIM1_PATH, "r", encoding="utf-8") as f:
        dim1 = json.load(f)
    true_by_doc_dim1 = dim1["true_by_doc"]
    pred_by_doc_dim1 = dim1["pred_by_doc"]
else:
    print(f"No se encontró {DIM1_PATH}. Evaluando gold set con el modelo fine-tuneado (chunking, window={WINDOW_WORDS})...")
    metrics_exact, true_by_doc, pred_by_doc = dimension_metrica_clasica(
        gold_examples, WINDOW_WORDS, OVERLAP_WORDS
    )
    true_by_doc_dim1 = {k: sorted(v) for k, v in true_by_doc.items()}
    pred_by_doc_dim1 = {k: sorted(v) for k, v in pred_by_doc.items()}
    resultado_dim1 = {
        "dimension": "metrica_clasica_exact_match",
        "modelo": BASE_CHECKPOINT,
        "adaptador_lora": str(MODEL_DIR),
        "gold_set": str(GOLD_SET_PATH),
        "n_ejemplos_evaluados": len(true_by_doc),
        "metrics": metrics_exact,
        "true_by_doc": true_by_doc_dim1,
        "pred_by_doc": pred_by_doc_dim1,
    }
    with open(DIM1_PATH, "w", encoding="utf-8") as f:
        json.dump(resultado_dim1, f, indent=2, ensure_ascii=False)
    print(f"Predicciones evaluadas y guardadas en {DIM1_PATH}")

rich_examples = []
for doc_id in sorted(true_by_doc_dim1):
    gold = sorted(true_by_doc_dim1[doc_id])
    pred = sorted(pred_by_doc_dim1.get(doc_id, []))
    rich_examples.append({
        "doc_id": doc_id,
        "n_gold": len(gold),
        "gold": gold,
        "pred": pred,
    })

print(f"Ejemplos cargados para el juez: {len(rich_examples)}")
print(f"Mismo gold set que exact-match y similitud semántica -- sin asimetría de subset.")

for doc in rich_examples[:3]:
    print(f"\ndoc_id: {doc['doc_id']} | n_gold: {doc['n_gold']}")
    print(f"  Gold: {doc['gold']}")
    print(f"  Pred: {doc['pred']}")

No se encontró /content/drive/MyDrive/TopicosIA/M2/outputs/resultado_dimension1.json. Evaluando gold set con el modelo fine-tuneado (chunking, window=277)...
Predicciones evaluadas y guardadas en /content/drive/MyDrive/TopicosIA/M2/outputs/resultado_dimension1.json
Ejemplos cargados para el juez: 59
Mismo gold set que exact-match y similitud semántica -- sin asimetría de subset.

doc_id: ex_0 | n_gold: 28
  Gold: ['adenopatías', 'adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales', 'adenopatías bilaterales ilíacas e inguinales', 'afectación de las cadenas ganglionares', 'brucelosis', 'carcinoma verrugoso de buscke-lowenstein', 'condiloma acuminado gigante de buschke-lowenstein', 'herida de la incisión de linfadenectomía', 'hipogonadismo hipergonadotrópico', 'infiltraba igualmente los tejidos pubianos, el escroto', 'infiltración de la grasa del tejido celular subcutáneo de la pared interna de los muslos ni de la grasa del periné', 'infiltración 

---
## 5. RÚBRICA del juez LLM

Evalúa en 4 dimensiones clínicas (1–5 cada una):

| Dimensión | Descripción |
|---|---|
| **Completitud** | ¿Capturó la mayoría de las enfermedades mencionadas? |
| **Exactitud de boundary** | ¿Los nombres coinciden con el gold (o casi)? |
| **Relevancia clínica** | ¿Las predicciones son términos clínicamente válidos? |
| **Ausencia de ruido** | ¿Evitó marcar términos que NO son enfermedades? |

Score final = promedio de las 4 dimensiones.

In [12]:
RUBRICA_SYSTEM = (
    "Eres un médico especialista en terminología médica y anotación de entidades clínicas. "
    "Tu tarea es evaluar la calidad de las entidades clínicas predichas por un sistema de NER, "
    "comparando sus predicciones con una lista gold. "
    "Evalúa SOLO el contenido. No sabes qué modelo generó las predicciones."
)

RUBRICA_TEMPLATE = """
# Evaluación de NER clínico — RUBRICA

## Referencia (gold standard)
Enfermedades correctas: {gold_str}

## Predicción del sistema
Enfermedades detectadas: {pred_str}

## Instrucciones
Evalúa en cada dimensión del 1 al 5:

1. **Completitud** (¿capturó la mayoría de las enfermedades gold?):
   5=todas/casi todas | 3=la mitad | 1=prácticamente nada

2. **Exactitud de boundary** (¿los nombres coinciden con el gold?):
   5=coincidencia exacta o diferencia mínima | 3=algunos coinciden, otros truncados | 1=no se parecen

3. **Relevancia clínica** (¿las predicciones son términos de enfermedades válidos?):
   5=todas válidas | 3=mezcla | 1=mayoría inválidas

4. **Ausencia de ruido** (¿evitó marcar términos que NO son enfermedades?):
   5=sin FP notables | 3=algunos FP | 1=demasiados FP

## Respuesta
Responde ÚNICAMENTE en JSON válido con el siguiente formato:
```json
{{
  "completitud": <1-5>,
  "exactitud_boundary": <1-5>,
  "relevancia_clinica": <1-5>,
  "ausencia_ruido": <1-5>,
  "justificacion": "<una oración breve>"
}}
```
"""


def build_judge_prompt(gold: list, pred: list, order: str = "normal") -> str:
    gold_str = ", ".join(gold) if gold else "(ninguna)"
    pred_str = ", ".join(pred) if pred else "(ninguna)"
    if order == "inverted":
        return RUBRICA_TEMPLATE.format(gold_str=pred_str, pred_str=gold_str)
    return RUBRICA_TEMPLATE.format(gold_str=gold_str, pred_str=pred_str)


def parse_judge_response(text: str) -> dict:
    # 1. Intentar parsear bloque JSON completo
    for pattern in [r"```(?:json)?\s*({[\s\S]*?})\s*```", r"({[\s\S]*?})"]:
        m = re.search(pattern, text)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                pass
    try:
        return json.loads(text.strip())
    except Exception:
        pass

    # 2. Fallback resiliente: extracción directa por regex de cada dimensión
    result = {}
    for key in ["completitud", "exactitud_boundary", "relevancia_clinica", "ausencia_ruido"]:
        val_m = re.search(rf'"{key}"\s*:\s*([1-5])', text, re.IGNORECASE)
        result[key] = int(val_m.group(1)) if val_m else None

    just_m = re.search(r'"justificacion"\s*:\s*"([^"\n]+)', text, re.IGNORECASE)
    result["justificacion"] = just_m.group(1) if just_m else (text[-200:].strip() if text else "Sin texto")
    return result


def compute_score(d: dict) -> float:
    vals = [d.get(k) for k in ["completitud", "exactitud_boundary",
                                "relevancia_clinica", "ausencia_ruido"]
            if d.get(k) is not None]
    return float(np.mean(vals)) if vals else None


def call_judge(gold: list, pred: list, order: str = "normal",
               retry: int = 5, sleep_s: float = 1.0) -> dict:
    """
    Llama al juez LLM vía Groq con reintentos y tolerancia a fallos.
    Si el modelo no soporta response_format={'type': 'json_object'}, hace fallback
    automático a modo texto estándar y parsea la respuesta con regex.
    Maneja también esperas exponenciales ante rate limits (429).
    """
    prompt = build_judge_prompt(gold, pred, order=order)
    use_json_mode = globals().get("SUPPORTS_JSON_MODE", True)

    for attempt in range(retry):
        try:
            kwargs = {
                "model": JUDGE_MODEL,
                "messages": [
                    {"role": "system", "content": RUBRICA_SYSTEM},
                    {"role": "user", "content": prompt},
                ],
                "temperature": JUDGE_TEMPERATURE,
                "max_tokens": JUDGE_MAX_TOKENS,
            }
            if use_json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            response = groq_client.chat.completions.create(**kwargs)
            time.sleep(sleep_s)
            content = response.choices[0].message.content
            parsed = parse_judge_response(content)

            # Si se obtuvo al menos una dimensión numérica válida, el llamado fue exitoso
            if any(parsed.get(k) is not None for k in ["completitud", "exactitud_boundary", "relevancia_clinica", "ausencia_ruido"]):
                return parsed

            # Si no parseó pero usábamos json_mode, reintentar sin json_mode
            if attempt == 0 and use_json_mode:
                use_json_mode = False
                continue

            return parsed
        except Exception as e:
            err_str = str(e)
            if "response_format" in err_str or "json_object" in err_str:
                use_json_mode = False

            is_rate_limit = "429" in err_str or "rate_limit" in err_str.lower()
            base_wait = 5 if is_rate_limit else 2
            wait_time = base_wait * (attempt + 1)
            print(f"    [Reintento {attempt+1}/{retry}] {e} -> esperando {wait_time}s...")
            time.sleep(wait_time)

    return {"completitud": None, "exactitud_boundary": None,
            "relevancia_clinica": None, "ausencia_ruido": None,
            "justificacion": "JUDGE_CALL_FAILED"}


test_res = call_judge(["asma crónica"], ["asma"])
print("Test call_judge verificado:", test_res)

Test call_judge verificado: {'completitud': 3, 'exactitud_boundary': 3, 'relevancia_clinica': 5, 'ausencia_ruido': 5, 'justificacion': "Capturó 'asma' correctamente pero omitió el modificador 'crónica', sin errores adicionales."}


---
## 6. Mitigación del sesgo de posición

### Qué es el sesgo de posición
Un LLM juez puede calificar de forma diferente dependiendo de si la predicción se presenta como "Referencia" o como "Predicción del sistema".

### Protocolo de mitigación
Evaluamos cada ejemplo dos veces:
1. **Orden normal:** gold = referencia, pred = predicción del sistema.
2. **Orden invertido:** pred = referencia, gold = predicción del sistema.

El score final mitigado es el **promedio de ambos**.

In [13]:
print(f"Corriendo juez (normal + invertido) sobre {len(rich_examples)} rich examples...")
print(f"Total llamadas al juez: {len(rich_examples) * 2}\n")

position_results = []

for i, doc in enumerate(rich_examples):
    print(f"  [{i+1}/{len(rich_examples)}] {doc['doc_id']} | n_gold={doc['n_gold']} | n_pred={len(doc['pred'])}")

    result_normal    = call_judge(doc["gold"], doc["pred"], order="normal")
    score_normal     = compute_score(result_normal)

    result_inverted  = call_judge(doc["gold"], doc["pred"], order="inverted")
    score_inverted   = compute_score(result_inverted)

    delta = abs(score_normal - score_inverted) if (score_normal is not None and score_inverted is not None) else None
    valid_scores = [s for s in [score_normal, score_inverted] if s is not None]
    score_mitigado = float(np.mean(valid_scores)) if valid_scores else None

    s_norm_str = f"{score_normal:.2f}" if score_normal is not None else "FAIL"
    s_inv_str  = f"{score_inverted:.2f}" if score_inverted is not None else "FAIL"
    s_mit_str  = f"{score_mitigado:.2f}" if score_mitigado is not None else "FAIL"
    print(f"      Score normal: {s_norm_str} | Score invertido: {s_inv_str} | Mitigado: {s_mit_str}")

    position_results.append({
        "doc_id"         : doc["doc_id"],
        "n_gold"         : doc["n_gold"],
        "n_pred"         : len(doc["pred"]),
        "gold"           : doc["gold"],
        "pred"           : doc["pred"],
        "score_normal"   : score_normal,
        "score_inverted" : score_inverted,
        "delta_posicion" : delta,
        "score_mitigado" : score_mitigado,
        "result_normal"  : result_normal,
        "result_inverted": result_inverted,
    })

print("\nListo.")

Corriendo juez (normal + invertido) sobre 59 rich examples...
Total llamadas al juez: 118

  [1/59] ex_0 | n_gold=28 | n_pred=24
      Score normal: 3.00 | Score invertido: FAIL | Mitigado: 3.00
  [2/59] ex_1 | n_gold=25 | n_pred=26
      Score normal: FAIL | Score invertido: FAIL | Mitigado: FAIL
  [3/59] ex_10 | n_gold=19 | n_pred=20
    [Reintento 1/5] Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m1s639c7e1prc7fnh0xsd944` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6362, Requested 2258. Please try again in 4.65s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}} -> esperando 5s...
      Score normal: 3.50 | Score invertido: 3.75 | Mitigado: 3.62
  [4/59] ex_11 | n_gold=19 | n_pred=19
      Score normal: 3.00 | Score invertido: 3.25 | Mitigado: 3.12
  [5/59] ex_12 | n_gold=19 | n_pred=19
      Score 

In [14]:
df_pos = pd.DataFrame(position_results)
deltas = df_pos["delta_posicion"].dropna()

print("=" * 55)
print("ANÁLISIS — SESGO DE POSICIÓN")
print("=" * 55)
print(f"  Ejemplos evaluados          : {len(df_pos)}")
print(f"  Delta medio  |normal-inv|   : {deltas.mean():.3f}")
print(f"  Delta máximo                : {deltas.max():.3f}")
print(f"  Delta mediana               : {deltas.median():.3f}")
print(f"  % ejemplos con delta > 1    : {(deltas > 1).mean()*100:.1f}%")
print()

if deltas.mean() < 0.5:
    verdict = "ESTABLE respecto a posición (delta medio < 0.5). Mitigación es una precaución razonable."
elif deltas.mean() < 1.0:
    verdict = "SESGO MODERADO. La mitigación por promedio es NECESARIA."
else:
    verdict = "SESGO FUERTE. El orden altera significativamente el juicio. Mitigación CRÍTICA."

print(f"Veredicto: {verdict}")

ANÁLISIS — SESGO DE POSICIÓN
  Ejemplos evaluados          : 59
  Delta medio  |normal-inv|   : 0.360
  Delta máximo                : 1.250
  Delta mediana               : 0.250
  % ejemplos con delta > 1    : 1.8%

Veredicto: ESTABLE respecto a posición (delta medio < 0.5). Mitigación es una precaución razonable.


---
## 7. Mitigación del sesgo de longitud

### Qué es el sesgo de longitud
El juez puede puntuar más alto respuestas más largas, independientemente de si son correctas.

### Protocolo de detección
Pares de validación con asimetría de longitud controlada:
- **Tipo A** — Correcta corta vs. Incorrecta larga: el juez debería puntuar más alto la correcta.
- **Tipo B** — Correcta larga vs. Incorrecta corta: el juez debería puntuar más alto la correcta.

In [15]:
LENGTH_BIAS_PAIRS = [
    # Tipo A — correcta corta vs. incorrecta larga
    {
        "tipo": "A", "descripcion": "Correcta corta vs. Incorrecta larga",
        "gold": ["neumonía", "sepsis"],
        "pred_correcta" : ["neumonía"],
        "pred_incorrecta": ["hiperglucemia", "dislipidemia", "hipotiroidismo",
                             "artritis reumatoide", "fibromialgia",
                             "esclerosis múltiple", "enfermedad de crohn", "psoriasis"],
    },
    {
        "tipo": "A", "descripcion": "Correcta corta vs. Incorrecta larga",
        "gold": ["infarto agudo de miocardio"],
        "pred_correcta" : ["infarto de miocardio"],
        "pred_incorrecta": ["gastritis crónica", "úlcera péptica", "reflujo gastroesofágico",
                             "colelitiasis", "pancreatitis aguda", "apendicitis",
                             "diverticulitis", "colitis ulcerosa", "hepatitis"],
    },
    {
        "tipo": "A", "descripcion": "Correcta corta vs. Incorrecta larga",
        "gold": ["carcinoma de células renales"],
        "pred_correcta" : ["carcinoma renal"],
        "pred_incorrecta": ["cefalea tensional", "migraña", "vértigo posicional",
                             "neuralgia del trigémino", "parálisis facial"],
    },
    # Tipo B — correcta larga vs. incorrecta corta
    {
        "tipo": "B", "descripcion": "Correcta larga vs. Incorrecta corta",
        "gold": ["diabetes mellitus", "hipertensión arterial", "dislipidemia", "obesidad"],
        "pred_correcta" : ["diabetes mellitus", "hipertensión arterial", "dislipidemia"],
        "pred_incorrecta": ["asma"],
    },
    {
        "tipo": "B", "descripcion": "Correcta larga vs. Incorrecta corta",
        "gold": ["neumonía adquirida en la comunidad", "derrame pleural", "insuficiencia respiratoria"],
        "pred_correcta" : ["neumonía adquirida en la comunidad", "derrame pleural"],
        "pred_incorrecta": ["fractura"],
    },
]

print(f"Pares de control de longitud: {len(LENGTH_BIAS_PAIRS)} pares definidos.")

Pares de control de longitud: 5 pares definidos.


In [16]:
print("Corriendo juez sobre pares de longitud...")
length_results = []

for i, pair in enumerate(LENGTH_BIAS_PAIRS):
    print(f"  [{i+1}/{len(LENGTH_BIAS_PAIRS)}] Tipo {pair['tipo']} — {pair['descripcion']}")
    r_cor = call_judge(pair["gold"], pair["pred_correcta"],  order="normal")
    r_inc = call_judge(pair["gold"], pair["pred_incorrecta"], order="normal")
    sc, si = compute_score(r_cor), compute_score(r_inc)
    length_results.append({
        "tipo"              : pair["tipo"],
        "descripcion"       : pair["descripcion"],
        "n_correcta"        : len(pair["pred_correcta"]),
        "n_incorrecta"      : len(pair["pred_incorrecta"]),
        "score_correcta"    : sc,
        "score_incorrecta"  : si,
        "juez_premia_calidad": (sc > si) if (sc is not None and si is not None) else None,
    })
    sc_str = f"{sc:.2f}" if sc is not None else "FAIL"
    si_str = f"{si:.2f}" if si is not None else "FAIL"
    ok_str = "OK" if (sc and si and sc > si) else "SESGO"
    print(f"      Score correcta: {sc_str} | Score incorrecta: {si_str} -> {ok_str}")

print("\nListo.")

Corriendo juez sobre pares de longitud...
  [1/5] Tipo A — Correcta corta vs. Incorrecta larga
      Score correcta: 4.50 | Score incorrecta: 3.00 -> OK
  [2/5] Tipo A — Correcta corta vs. Incorrecta larga
      Score correcta: 4.00 | Score incorrecta: 3.00 -> OK
  [3/5] Tipo A — Correcta corta vs. Incorrecta larga
      Score correcta: 4.50 | Score incorrecta: 3.00 -> OK
  [4/5] Tipo B — Correcta larga vs. Incorrecta corta
      Score correcta: 4.75 | Score incorrecta: 3.00 -> OK
  [5/5] Tipo B — Correcta larga vs. Incorrecta corta
    [Reintento 1/5] Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m1s639c7e1prc7fnh0xsd944` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6173, Requested 1992. Please try again in 1.2375s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}} -> esperando 5s...
      Score correct

In [17]:
df_len  = pd.DataFrame(length_results)
n_ok    = df_len["juez_premia_calidad"].sum()
n_total = df_len["juez_premia_calidad"].notna().sum()

print("=" * 55)
print("ANÁLISIS — SESGO DE LONGITUD")
print("=" * 55)
print(f"  Pares evaluados: {n_total}")
print(f"  El juez premió calidad sobre longitud: {n_ok}/{n_total} ({n_ok/n_total*100:.0f}%)")
print()

if n_ok / n_total >= 0.8:
    print("El juez NO muestra sesgo de longitud relevante.")
elif n_ok / n_total >= 0.6:
    print("Sesgo LEVE de longitud — en algunos casos premia extensión.")
else:
    print("Sesgo FUERTE de longitud — agregar instrucción explícita en la rúbrica.")

print()
print(df_len[["tipo", "n_correcta", "n_incorrecta", "score_correcta", "score_incorrecta", "juez_premia_calidad"]])

ANÁLISIS — SESGO DE LONGITUD
  Pares evaluados: 5
  El juez premió calidad sobre longitud: 5/5 (100%)

El juez NO muestra sesgo de longitud relevante.

  tipo  n_correcta  n_incorrecta  score_correcta  score_incorrecta  \
0    A           1             8            4.50               3.0   
1    A           1             9            4.00               3.0   
2    A           1             5            4.50               3.0   
3    B           3             1            4.75               3.0   
4    B           2             1            4.75               1.0   

   juez_premia_calidad  
0                 True  
1                 True  
2                 True  
3                 True  
4                 True  


---
## 8. Sesgo de auto-preferencia (documentación)

### Qué es
Un LLM juez tiende a preferir outputs de modelos de su misma familia arquitectural o de entrenamiento.

### Situación en este proyecto
- **Juez:** Modelo externo vía Groq (`openai/gpt-oss-120b`).
- **Modelo evaluado:** `roberta-base-biomedical-clinical-es` (encoder RoBERTa, PlanTL / BSC).
- **Modelo alternativo del proyecto:** `mT5` (encoder-decoder, Google).

Al usar un juez LLM externo vía Groq, el juez **no pertenece a la familia de ninguno de los modelos del proyecto**, lo cual mitiga el sesgo de auto-preferencia por diseño.

### Protocolo de mitigación activo
1. **Anonimización total:** El prompt del juez nunca menciona el modelo que generó las predicciones.
2. **Formato idéntico:** Tanto gold como predicciones se presentan como listas de strings.

In [18]:
sample_prompt = build_judge_prompt(["neumonía", "sepsis"], ["neumonía"])
forbidden     = ["roberta", "bert", "mt5", "llama", "gpt", "gemini", "plantl", "biomedical", "clinical"]
found = [n for n in forbidden if n.lower() in sample_prompt.lower()]

if not found:
    print("El prompt NO menciona ningún modelo — output completamente anonimizado.")
else:
    print(f"ATENCIÓN: el prompt menciona {found} — revisar RUBRICA_TEMPLATE.")

print()
print("Resumen de mitigaciones de auto-preferencia:")
print("  1. Output anonimizado (sin nombres de modelo en el prompt): OK")
print("  2. Formato estandarizado (listas de strings para todos los modelos): OK")
print("  3. Juez externo a las familias evaluadas (Groq LLM vs RoBERTa/mT5): OK")

El prompt NO menciona ningún modelo — output completamente anonimizado.

Resumen de mitigaciones de auto-preferencia:
  1. Output anonimizado (sin nombres de modelo en el prompt): OK
  2. Formato estandarizado (listas de strings para todos los modelos): OK
  3. Juez externo a las familias evaluadas (Groq LLM vs RoBERTa/mT5): OK


---
## 9. Scorecard final sobre rich examples

Usamos el **score mitigado** (promedio normal + invertido, Sección 6) como score final del juez.

In [19]:
def compute_exact_f1_doc(gold: list, pred: list) -> dict:
    """F1 exact-match para un documento (para triangular con la métrica clásica)."""
    g, p = set(gold), set(pred)
    tp   = len(g & p)
    fp   = len(p - g)
    fn   = len(g - p)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return {"precision": prec, "recall": rec, "f1": f1}


scorecard_rows = []
for row in position_results:
    exact = compute_exact_f1_doc(row["gold"], row["pred"])
    scorecard_rows.append({
        "doc_id"        : row["doc_id"],
        "n_gold"        : row["n_gold"],
        "n_pred"        : row["n_pred"],
        "f1_exacto"     : round(exact["f1"], 3),
        "prec_exacta"   : round(exact["precision"], 3),
        "rec_exacto"    : round(exact["recall"], 3),
        "score_juez"    : round(row["score_mitigado"], 3) if row["score_mitigado"] is not None else None,
        "delta_posicion": round(row["delta_posicion"], 3) if row["delta_posicion"] is not None else None,
        "gold"          : row["gold"],
        "pred"          : row["pred"],
    })

df_sc = pd.DataFrame(scorecard_rows)

print("=" * 60)
print("SCORECARD FINAL — JUEZ LLM (SCORE MITIGADO) vs. F1 EXACTO")
print("=" * 60)
print(f"Documentos evaluados : {len(df_sc)}")
print(f"F1 exacto medio      : {df_sc['f1_exacto'].mean():.3f}")
print(f"Score juez medio     : {df_sc['score_juez'].mean():.3f} / 5.0")
print(f"Correlación Pearson  : {df_sc['f1_exacto'].corr(df_sc['score_juez']):.3f}")
print()
print(df_sc[["doc_id", "n_gold", "n_pred", "f1_exacto", "score_juez", "delta_posicion"]].to_string())

SCORECARD FINAL — JUEZ LLM (SCORE MITIGADO) vs. F1 EXACTO
Documentos evaluados : 59
F1 exacto medio      : 0.718
Score juez medio     : 3.983 / 5.0
Correlación Pearson  : 0.660

   doc_id  n_gold  n_pred  f1_exacto  score_juez  delta_posicion
0    ex_0      28      24      0.654       3.000             NaN
1    ex_1      25      26      0.706         NaN             NaN
2   ex_10      19      20      0.615       3.625            0.25
3   ex_11      19      19      0.526       3.125            0.25
4   ex_12      19      19      0.789       4.125            0.25
5   ex_13      18      19      0.865       4.250            0.00
6   ex_14      17      16      0.667       3.875            0.75
7   ex_15      17      19      0.667       3.375            0.25
8   ex_16      16      17      0.788       3.625            0.75
9   ex_17      15      13      0.643       3.125            0.25
10  ex_18      15      13      0.714       3.750            0.50
11  ex_19      15      13      0.857      

---
## 10. Interpretación cualitativa — ¿Qué revela el juez que el F1 exacto no capta?

| Patrón | F1 exacto | Score juez | Interpretación |
|--------|-----------|-----------|----------------|
| **A** | ≈ 0 | ≥ 3.5 | Entendió la entidad pero con **boundary distinto** — error de span, no de comprensión. |
| **B** | > 0.3 | ≤ 2.5 | Acertó la string exacta pero el output **no es clínicamente satisfactorio**. |
| **C** | ≈ 0 | ≤ 2.0 | Fallo completo. |
| **D** | ≥ 0.7 | ≥ 4.0 | Funcionamiento correcto — referencia positiva. |
| **E** | resto | resto | Caso mixto. |

In [20]:
def classify_pattern(row):
    s, f = row["score_juez"], row["f1_exacto"]
    if s is None or f is None: return "? — datos faltantes"
    if f < 0.1 and s >= 3.5:   return "A — boundary error (comprensión OK, span malo)"
    if f > 0.3 and s <= 2.5:   return "B — string OK, calidad clínica baja"
    if f < 0.1 and s <= 2.0:   return "C — fallo completo"
    if f >= 0.7 and s >= 4.0:  return "D — referencia positiva"
    return "E — caso mixto"


df_sc["patron"] = df_sc.apply(classify_pattern, axis=1)
print("Distribución de patrones:")
print(df_sc["patron"].value_counts().to_string())

for patron_id in ["A", "B", "C", "D"]:
    subset = df_sc[df_sc["patron"].str.startswith(patron_id)]
    if not subset.empty:
        ej = subset.iloc[0]
        print(f"\nEjemplo Patrón {patron_id} ({ej['doc_id']}):")
        print(f"  F1 exacto: {ej['f1_exacto']} | Score juez: {ej['score_juez']}")
        print(f"  Gold     : {ej['gold']}")
        print(f"  Pred     : {ej['pred']}")

Distribución de patrones:
patron
E — caso mixto             35
D — referencia positiva    24

Ejemplo Patrón D (ex_12):
  F1 exacto: 0.789 | Score juez: 4.125
  Gold     : ['ascitis, secundaria a peritonitis por echerichia coli', 'calcificación peritoneal', 'esclerosis y calcificación de toda la membrana peritoneal', 'eventración peritoneal en el orificio del catéter', 'hiperparatiroidismo grave', 'hipogonadismo', 'infección de túnel por el estafilococo', 'insuficiencia renal', 'isquemia aguda en miembro inferior', 'obesidad', 'obstrucción femoropoplítea', 'peritonitis', 'peritonitis por echerichia coli', 'peritonitis, causada por dos gérmenes (staphilococcus epidermidis y pseudomona aeruginosa', 'polidactilia', 'retinitis pigmentaria', 'retraso mental', 'síndrome de laurence-moon-biell', 'trombosis venosa']
  Pred     : ['ascitis', 'calcificación peritoneal', 'esclerosis y calcificación de toda la membrana peritoneal', 'eventración peritoneal en el orificio del catéter', 'hiperparatir

In [21]:
import json as _j

OUTPUT_DIR = PROJECT_ROOT / "M2" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "llm_judge_scorecard.csv"
df_sc.drop(columns=["gold", "pred"], errors="ignore").to_csv(
    csv_path, index=False, encoding="utf-8"
)
print(f"Scorecard  -> {csv_path}")

bias_summary = {
    "judge_model"        : JUDGE_MODEL,
    "judge_provider"     : "Groq",
    "base_model"         : BASE_CHECKPOINT,
    "seed"               : SEED,
    "n_rich_examples"    : len(df_sc),
    "score_juez_mean"    : float(df_sc["score_juez"].mean()) if not df_sc["score_juez"].empty else 0.0,
    "score_juez_std"     : float(df_sc["score_juez"].std()) if not df_sc["score_juez"].empty else 0.0,
    "f1_exacto_mean"     : float(df_sc["f1_exacto"].mean()) if not df_sc["f1_exacto"].empty else 0.0,
    "position_bias": {
        "delta_mean" : float(deltas.mean()) if not deltas.empty else 0.0,
        "delta_max"  : float(deltas.max()) if not deltas.empty else 0.0,
        "mitigation" : "average of normal and inverted order scores",
    },
    "length_bias": {
        "pairs_quality_wins": int(n_ok),
        "pairs_total"       : int(n_total),
        "pct_quality_wins"  : float(n_ok / n_total) if n_total > 0 else 0.0,
        "mitigation"        : "manual validation pairs — judge follows content, not length",
    },
    "auto_preference_bias": {
        "output_anonymized"   : True,
        "format_standardized" : True,
        "cross_family_test"   : False,
        "note": "Cannot quantify without a judge from a different model family",
    },
    "pattern_distribution": df_sc["patron"].value_counts().to_dict(),
}

json_path = OUTPUT_DIR / "llm_judge_bias_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    _j.dump(bias_summary, f, indent=2, ensure_ascii=False)
print(f"Bias summary -> {json_path}")

print("\n" + "=" * 55)
print("RESUMEN FINAL")
print("=" * 55)
print(f"  Sesgo posición  — delta medio: {deltas.mean():.3f}  | mitigación: promedio normal+invertido")
print(f"  Sesgo longitud  — calidad gana: {n_ok}/{n_total}    | mitigación: pares de validación")
print(f"  Auto-preferencia— anonimizado: OK | test cross-family: no disponible (limitación)")
print(f"  Score juez final (mitigado)  : {df_sc['score_juez'].mean():.3f}")
print(f"  F1 exacto (mismo subset)     : {df_sc['f1_exacto'].mean():.3f}")

Scorecard  -> /content/drive/MyDrive/TopicosIA/M2/outputs/llm_judge_scorecard.csv
Bias summary -> /content/drive/MyDrive/TopicosIA/M2/outputs/llm_judge_bias_summary.json

RESUMEN FINAL
  Sesgo posición  — delta medio: 0.360  | mitigación: promedio normal+invertido
  Sesgo longitud  — calidad gana: 5/5    | mitigación: pares de validación
  Auto-preferencia— anonimizado: OK | test cross-family: no disponible (limitación)
  Score juez final (mitigado)  : 3.983
  F1 exacto (mismo subset)     : 0.718
